# Phase 2: Relation聚类分析

将Phase 1提取的128个relation聚类到15-20个标准relation

## 测试两种方法：
1. BERTopic（自动探索 + 可视化）
2. Agglomerative Clustering（精确控制）

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.clustering.relation_clusterer import RelationClusterer

# 设置绘图样式
plt.rcParams['figure.figsize'] = (14, 8)
sns.set_style('whitegrid')

## 1. 加载Phase 1数据

In [ ]:
# 初始化聚类器
clusterer = RelationClusterer(
    phase1_result_path='../results/phase1_5percent_exploration.json'
)

# 加载数据
phase1_data = clusterer.load_phase1_results()

# 提取relations
all_relations, relation_counts = clusterer.extract_relations(phase1_data)

## 2. 生成BGE Embeddings

In [ ]:
# 生成embeddings（两种方法都需要）
embeddings = clusterer.embed_relations_bge(model_name="BAAI/bge-base-en-v1.5")

print(f"\nEmbedding维度: {embeddings.shape}")

## 3. 方法1: BERTopic聚类

In [ ]:
# BERTopic聚类
topic_model, topics, probs = clusterer.cluster_with_bertopic(
    min_cluster_size=3,
    n_components_umap=10,
    verbose=True
)

In [ ]:
# 可视化BERTopic结果
try:
    # Topic分布图
    fig1 = topic_model.visualize_topics()
    fig1.show()
    
    # 层次聚类图
    fig2 = topic_model.visualize_hierarchy()
    fig2.show()
    
    # Topic词云（如果有的话）
    fig3 = topic_model.visualize_barchart(top_n_topics=10)
    fig3.show()
except Exception as e:
    print(f"可视化错误（预期，因为我们的数据是短词组）: {e}")

In [ ]:
# 查看BERTopic的topic信息
topic_info = topic_model.get_topic_info()
print("\nBERTopic Topics:")
print(topic_info[['Topic', 'Count', 'Name']])

# 查看每个topic包含的relations
print("\n每个Topic的Relations:")
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    topic_relations = [
        clusterer.unique_relations[i]
        for i, t in enumerate(topics)
        if t == topic_id
    ]
    print(f"\nTopic {topic_id}:")
    for rel in topic_relations:
        print(f"  - {rel:30s} (count: {relation_counts[rel]})")

## 4. 方法2: Agglomerative Clustering

In [ ]:
# 测试不同的n_clusters
test_n_clusters = [15, 18, 20]

results_agg = {}

for n_clusters in test_n_clusters:
    print(f"\n{'='*60}")
    print(f"测试 n_clusters = {n_clusters}")
    print(f"{'='*60}")
    
    labels, model = clusterer.cluster_with_agglomerative(
        n_clusters=n_clusters,
        linkage='average'
    )
    
    results_agg[n_clusters] = {
        'labels': labels,
        'model': model
    }

## 5. 可视化Agglomerative聚类结果

In [ ]:
# 使用层次聚类树状图
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

# 计算距离矩阵
distance_matrix = pdist(embeddings, metric='cosine')
linkage_matrix = linkage(distance_matrix, method='average')

# 绘制树状图
plt.figure(figsize=(20, 10))
dendrogram(
    linkage_matrix,
    labels=clusterer.unique_relations,
    leaf_rotation=90,
    leaf_font_size=8
)
plt.title('Hierarchical Clustering Dendrogram (Average Linkage, Cosine Distance)')
plt.xlabel('Relations')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

print("\n💡 从树状图可以看出合理的切割点（红色水平线的位置）")

In [ ]:
# 用UMAP降维到2D可视化
from umap import UMAP

# UMAP降维
umap_model = UMAP(n_components=2, random_state=42, metric='cosine')
embeddings_2d = umap_model.fit_transform(embeddings)

# 绘制不同n_clusters的结果
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, n_clusters in enumerate(test_n_clusters):
    ax = axes[idx]
    labels = results_agg[n_clusters]['labels']
    
    scatter = ax.scatter(
        embeddings_2d[:, 0],
        embeddings_2d[:, 1],
        c=labels,
        cmap='tab20',
        s=100,
        alpha=0.6
    )
    
    ax.set_title(f'Agglomerative Clustering (n={n_clusters})')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # 添加部分标签（高频relations）
    top_relations = [rel for rel, _ in relation_counts.most_common(15)]
    for i, rel in enumerate(clusterer.unique_relations):
        if rel in top_relations:
            ax.annotate(
                rel,
                (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                fontsize=8,
                alpha=0.7
            )

plt.tight_layout()
plt.show()

## 6. 选择最佳方案并生成映射

In [ ]:
# 根据上面的可视化结果，选择最佳的n_clusters
# 这里先用18作为示例
BEST_N_CLUSTERS = 18

print(f"选择方案: Agglomerative Clustering with n_clusters={BEST_N_CLUSTERS}")

# 生成relation映射
best_labels = results_agg[BEST_N_CLUSTERS]['labels']
relation_mapping = clusterer.generate_relation_mapping(
    labels=best_labels,
    method='frequency'  # 选择频次最高的作为标准名
)

In [ ]:
# 查看映射详情
print("\nRelation映射详情:")
print("="*80)

# 按标准relation分组显示
standard_relations = sorted(set(relation_mapping.values()))

for std_rel in standard_relations:
    # 找到映射到该标准relation的所有原始relations
    mapped_relations = [
        (orig_rel, relation_counts[orig_rel])
        for orig_rel, std in relation_mapping.items()
        if std == std_rel
    ]
    mapped_relations.sort(key=lambda x: x[1], reverse=True)
    
    total_count = sum(count for _, count in mapped_relations)
    
    print(f"\n{std_rel} (总计: {total_count} instances):")
    for orig_rel, count in mapped_relations:
        if orig_rel == std_rel:
            print(f"  ★ {orig_rel:30s}: {count:4d}  [标准名]")
        else:
            print(f"    {orig_rel:30s}: {count:4d}")

## 7. 保存结果

In [ ]:
# 保存Agglomerative聚类结果
clusterer.save_results(
    relation_mapping=relation_mapping,
    output_path='../results/relation_mapping_agglomerative.json',
    method='agglomerative',
    metadata={
        'n_clusters': BEST_N_CLUSTERS,
        'linkage': 'average',
        'embedding_model': 'BAAI/bge-base-en-v1.5'
    }
)

print("\n✅ Relation聚类完成！")
print(f"   标准relation数: {len(set(relation_mapping.values()))}")
print(f"   映射文件: results/relation_mapping_agglomerative.json")

In [ ]:
# 保存最终的Relation映射
clusterer.save_results(
    relation_mapping=final_relation_mapping,
    output_path='../results/relation_mapping_final.json',
    method='agglomerative_finalized',
    metadata={
        'n_clusters': FINAL_N,
        'min_total_instances_threshold': MIN_INSTANCES_THRESHOLD,
        'linkage': 'average',
        'embedding_model': 'BAAI/bge-base-en-v1.5',
        'finalization_date': '2026-01-02'
    }
)

print("\n" + "="*80)
print("✅ Phase 2 Relation聚类完成！")
print("="*80)
print(f"   原始relation数: {len(relation_counts)}")
print(f"   最终标准relation数: {len(standard_relations)}")
print(f"   映射文件: results/relation_mapping_final.json")
print(f"\n下一步: Phase 2b - Entity聚类 (626个entity → 200-300个)")
print("="*80)"